In [ ]:
import os
from pathlib import Path
import tempfile
import shutil

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf

import hvplot.pandas
from bokeh.io import output_notebook
output_notebook()

teehr.__version__

In [ ]:
spark = create_spark_session()
ev = teehr.RemoteReadOnlyEvaluation(spark=spark)

In [ ]:
USGS_GAGE_ID = "usgs-01010070"
PRIMARY_CONFIGURATION_NAME = "usgs_observations"
SECONDARY_CONFIGURATION_NAME = "nwm30_retrospective"

events_df = ev.table("sim_joined_timeseries").filter([
    {
        "column": "configuration_name",
        "operator": "=",
        "value": SECONDARY_CONFIGURATION_NAME
    },
    {
        "column": "primary_location_id",
        "operator": "=",
        "value": USGS_GAGE_ID
    }
]).add_calculated_fields([
    tcf.AbovePercentileEventDetection()
]).to_pandas()

pt = events_df.hvplot(
    x="value_time",
    y="primary_value",
    by="event_above_id",
    label="USGS Observations",
)
pt.opts(
    title="Observed Events Over 85th Percentile", 
    width=500, height=400, 
    show_legend=False
)

In [ ]:
spark.stop()